In [16]:
# Install if needed
!pip install rdflib
!pip install networkx

In [10]:
# Import von rdflib, networkx und andere libraries
from rdflib import Graph, Namespace, RDF
import networkx as nx
from collections import deque
import matplotlib.pyplot as plt

In [11]:
# === 4. BFS with class filtering ===
def bfs_with_class_filter(G, start_node, target_class_uri):
    visited = set()
    queue = deque([start_node])
    matched_nodes = []

    while queue:
        node = queue.popleft()
        if node in visited:
            continue
        visited.add(node)

        if G.nodes[node].get('class') == target_class_uri:
            matched_nodes.append(node)

        for neighbor in G.neighbors(node):
            if neighbor not in visited:
                queue.append(neighbor)

    return matched_nodes

In [13]:
# Gerichtete Graph erstellen
def graph_from_rdf(g):
    
    G = nx.DiGraph()
    
    # Extract rdf:type information
    types = {}
    for s, p, o in g:
        if p == RDF.type:
            types[str(s)] = str(o)
    
    # Add all triples except rdf:type as edges
    for s, p, o in g:
        s_str, o_str, p_str = str(s), str(o), str(p)
        G.add_edge(s_str, o_str, label=p_str)
    
    # Add node classes as attributes
    for node in G.nodes:
        G.nodes[node]['class'] = types.get(node, None)

    # === 3. Show the entire KG ===
    print("Knowledge Graph:")
    for u, v, d in G.edges(data=True):
        label = d['label'].split("/")[-1]
        print(f"{u.split('/')[-1]} --[{label}]--> {v.split('/')[-1]}")

    
    return G

In [ ]:
# SCHRITT 1: Knowledge Graph vollständig aufbauen
from rdflib import Literal
from rdflib.namespace import XSD

g = Graph()
SC = Namespace("http://example.org/")

# --- Typ-Triples (rdf:type) ---
g.add((SC.supplyChain1,          RDF.type, SC.SupplyChain))
g.add((SC.teilzulieferer,        RDF.type, SC.Firma))
g.add((SC.hersteller,            RDF.type, SC.Firma))
g.add((SC.einzelhaendler1,       RDF.type, SC.Firma))
g.add((SC.einzelhaendler2,       RDF.type, SC.Firma))
g.add((SC.bestand,               RDF.type, SC.ValueStream))
g.add((SC.produktion,            RDF.type, SC.ValueStream))
g.add((SC.verkauf1,              RDF.type, SC.ValueStream))
g.add((SC.verkauf2,              RDF.type, SC.ValueStream))
g.add((SC.vorlaufzeitBestand,    RDF.type, SC.Vorlaufzeit))
g.add((SC.vorlaufzeitProduktion, RDF.type, SC.Vorlaufzeit))
g.add((SC.vorlaufzeitVerkauf1,   RDF.type, SC.Vorlaufzeit))
g.add((SC.vorlaufzeitVerkauf2,   RDF.type, SC.Vorlaufzeit))

# --- Struktur-Triples (istTeilVon) ---
# Firmen sind Teil der Supply Chain
g.add((SC.teilzulieferer,  SC.istTeilVon, SC.supplyChain1))
g.add((SC.hersteller,      SC.istTeilVon, SC.supplyChain1))
g.add((SC.einzelhaendler1, SC.istTeilVon, SC.supplyChain1))
g.add((SC.einzelhaendler2, SC.istTeilVon, SC.supplyChain1))

# Value Streams gehoeren zu ihrer jeweiligen Firma
g.add((SC.bestand,    SC.istTeilVon, SC.teilzulieferer))
g.add((SC.produktion, SC.istTeilVon, SC.hersteller))
g.add((SC.verkauf1,   SC.istTeilVon, SC.einzelhaendler1))
g.add((SC.verkauf2,   SC.istTeilVon, SC.einzelhaendler2))

# Vorlaufzeiten gehoeren zu ihrem jeweiligen Value Stream
g.add((SC.vorlaufzeitBestand,    SC.istTeilVon, SC.bestand))
g.add((SC.vorlaufzeitProduktion, SC.istTeilVon, SC.produktion))
g.add((SC.vorlaufzeitVerkauf1,   SC.istTeilVon, SC.verkauf1))
g.add((SC.vorlaufzeitVerkauf2,   SC.istTeilVon, SC.verkauf2))

# --- Dauer-Triples (Vorlaufzeiten in Tagen) ---
g.add((SC.vorlaufzeitBestand,    SC.dauer, Literal(5,  datatype=XSD.integer)))
g.add((SC.vorlaufzeitProduktion, SC.dauer, Literal(10, datatype=XSD.integer)))
g.add((SC.vorlaufzeitVerkauf1,   SC.dauer, Literal(2,  datatype=XSD.integer)))
g.add((SC.vorlaufzeitVerkauf2,   SC.dauer, Literal(3,  datatype=XSD.integer)))

print(f"KG aufgebaut mit {len(g)} Triples.")

In [ ]:
# SCHRITT 2

G = graph_from_rdf(g)

In [ ]:
# SCHRITT 3: BFS – Gesamtlieferzeit berechnen

start_node = str(SC.supplyChain1)
target_class = str(SC.ValueStream)

# Rückwarts traversal BFS
rwarts_G = G.reverse()

results = bfs_with_class_filter(rwarts_G, start_node, target_class)

print("BFS Result (full URIs):", results)

# --- Fortsetzung: Vorlaufzeiten je ValueStream ermitteln und summieren ---
print("\nVorlaufzeiten je ValueStream:")
gesamt = 0
for vs_node in results:
    # Im umgekehrten Graph zeigen Vorlaufzeit-Knoten auf ValueStream-Knoten
    for nachbar in rwarts_G.neighbors(vs_node):
        if G.nodes[nachbar].get('class') == str(SC.Vorlaufzeit):
            # Dauer aus dem Originalgraph lesen (Kante: VZ --[dauer]--> Literal)
            for dauer_knoten, edata in G[nachbar].items():
                if edata.get("label") == str(SC.dauer):
                    dauer = int(dauer_knoten)
                    gesamt += dauer
                    print(f"  {vs_node.split('/')[-1]} <- {nachbar.split('/')[-1]}: {dauer} Tage")

print(f"\nGesamtlieferzeit: {gesamt} Tage")

In [ ]:
# SCHRITT 4: Optimierung – neue Triples für direkten Zugriff
#
# Problem: BFS muss 3 Ebenen tief traversieren:
#   SC1 → Firma → ValueStream → Vorlaufzeit
#
# Lösung: Direkte "hatVorlaufzeit"-Kanten von supplyChain1 zu den Vorlaufzeit-Instanzen.
#   SC1 → Vorlaufzeit  (Tiefe 1 statt Tiefe 3)

g.add((SC.supplyChain1, SC.hatVorlaufzeit, SC.vorlaufzeitBestand))
g.add((SC.supplyChain1, SC.hatVorlaufzeit, SC.vorlaufzeitProduktion))
g.add((SC.supplyChain1, SC.hatVorlaufzeit, SC.vorlaufzeitVerkauf1))
g.add((SC.supplyChain1, SC.hatVorlaufzeit, SC.vorlaufzeitVerkauf2))

print("Neue Triples hinzugefügt: supplyChain1 --[hatVorlaufzeit]--> Vorlaufzeit-Instanzen")

# Optimierten Graph neu aufbauen
G2 = graph_from_rdf(g)

# Optimierter BFS: vorwärts ab SC1, findet Vorlaufzeiten direkt in Tiefe 1
vz_opt = bfs_with_class_filter(G2, str(SC.supplyChain1), str(SC.Vorlaufzeit))

print("\nOptimierter BFS – gefundene Vorlaufzeiten:")
gesamt_opt = 0
for vz in vz_opt:
    for nachbar, edata in G2[vz].items():
        if edata.get("label") == str(SC.dauer):
            dauer = int(nachbar)
            gesamt_opt += dauer
            print(f"  {vz.split('/')[-1]}: {dauer} Tage")

print(f"\nGesamtlieferzeit: {gesamt_opt} Tage")
print("\nVorher: 3 BFS-Ebenen (SC1 -> Firma -> ValueStream -> Vorlaufzeit)")
print("Nachher: 1 BFS-Ebene  (SC1 -> Vorlaufzeit)")